# Introducción
El objetivo principal de esta actividad es diseñar, implementar y evaluar un modelos de Aprendizaje Supervisado y No Supervisado sobre un entorno de Big Data, utilizando PySpark. Tomando como base el conjunto masivo de datos seleccionado para nuestro proyecto(2019-Oct.csv), el cual es de un comercio electrónico que registra millones de interacciones de usuarios. El proyecto busca resolver problemáticas estratégicas de negocio mediante dos enfoques:
1. La predicción del comportamiento de compra a través de modelos supervisados
2. La identificación de segmentos de mercado mediante modelos no supervisados

Aprendizaje Supervisado: Cuando entrenamos el modelo utilizando un conjunto de datos etiquetados. En otraspalabras la máquina conoce la respuesta correcta.  
En esta actividad implementamos Random Forest Classifier para predecir si un usuario realizará una compra (1) o si solo visualizará un producto (0) basándose en el precio y la marca.

Aprendizaje No Supervisado: Aquí el modelo trabaja con datos no etiquetadospor lo que busca patrones, anomalías o agrupaciones.
En esta actividad implementamos K-Means Clustering para agrupar las interacciones de los usuarios en 3 perfiles de clientes según su comportamiento económico.

Conectamos con Drive e importamos Pyspark


In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install pyspark

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Importamos la librerias necesarias

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, split

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import StandardScaler

from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.sql.functions import when

In [3]:
spark = SparkSession.builder.appName("Exploracion_ECommerce").getOrCreate()

ruta = "/content/drive/MyDrive/Big Data/2019-Oct.csv"
df_crudo = spark.read.csv(ruta, header=True, inferSchema=True)

print("=== ESQUEMA DE LA BASE DE DATOS ===")
df_crudo.printSchema()

print("\n=== MUESTRA DE LOS PRIMEROS 5 REGISTROS ===")
df_crudo.show(5, truncate=False)

=== ESQUEMA DE LA BASE DE DATOS ===
root
 |-- event_time: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- category_id: long (nullable = true)
 |-- category_code: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- user_session: string (nullable = true)


=== MUESTRA DE LOS PRIMEROS 5 REGISTROS ===
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|event_time         |event_type|product_id|category_id        |category_code                      |brand   |price  |user_id  |user_session                        |
+-------------------+----------+----------+-------------------+-----------------------------------+--------+-------+---------+------------------------------------+
|2019-10-01 00:00:00|view      |44600062  |2103

# Preparación y selección de datos

En nuestra base de datos global, aproximadamente el 98% de las interacciones corresponden a usuarios que solo observan productos o los abandonan en el carrito, mientras que apenas un margen cercano al 2% representa compras. Para evitar disparidad extrema, implementamos un Muestreo Estratificado mediante la función .sampleBy(). Esta técnica divide intencionalmente la base de datos en subgrupos según nuestra variable objetivo, permitiéndonos extraer observaciones de forma independiente y controlada de cada uno de ellos. La configuración de las fracciones analíticas elegidas ({0.0: 0.01, 1.0: 0.10}).

Fracción para No Compradores (0.0: 0.01): Al ser el grupo masivo, extraer únicamente el 1% es estadísticamente suficiente para obtener una base sólida de 241,791 registros, capturando perfectamente la diversidad de precios y marcas de los usuarios.

Fracción para Compradores (1.0: 0.10): Al ser la clase minoritaria y el de mayor interés para el negocio, extraemos una proporción  de 10.

Limpieza y Filtrado: Se eliminaron de forma estricta los valores nulos (dropna) en las columnas fundamentales (brand, category_code, price) para prevenir errores. Utilizamos StringIndexer para la codificación categórica, ya que, los algoritmos de Machine Learning solo entienden variables numéricas. Las columnas de texto de alta variedad como la marca (brand) y el código de categoría (category_code) fueron transformadas en índices numéricos ordenandolos por frecuencia.

In [4]:
df_etiquetado = df_crudo.withColumn("compro",
                                    when(col("event_type") == "purchase", 1.0).otherwise(0.0))

# Filtrar datos nulos
df_limpio = df_etiquetado.dropna(subset=["brand", "category_code", "price"])

# Muestreo Estratificado
fracciones = {0.0: 0.01, 1.0: 0.10}
df_muestra = df_limpio.sampleBy("compro", fracciones, seed=42)

print(f"Total en la muestra: {df_muestra.count()} registros.")
df_muestra.select("event_type", "brand", "category_code", "price", "compro").show(5)

Total en la muestra: 315392 registros.
+----------+-------+--------------------+------+------+
|event_type|  brand|       category_code| price|compro|
+----------+-------+--------------------+------+------+
|      view|    fly|electronics.smart...| 43.73|   0.0|
|      view|  apple|electronics.smart...|464.13|   0.0|
|      view|   asus|computers.compone...| 192.8|   0.0|
|      view|samsung|electronics.smart...|241.19|   0.0|
|      view| xiaomi|electronics.smart...|142.86|   0.0|
+----------+-------+--------------------+------+------+
only showing top 5 rows


# Aprendizaje supervisado
Utilizamos **Random Forest Classifier** porque es excelente para combinar datos de texto y números (como marcas, categorías y precios) y detectar patrones de compra complejos. Para que funcionara correctamente con nuestro volumen de Big Data, aplicamos tres ajustes clave:
1. Configuramos **`maxBins=1500`** para solucionar un error del sistema y permitir que Spark pudiera procesar las 1,150 marcas únicas sin colapsar
2. Usamos **`ParamGridBuilder`** para poner a competir combinaciones de árboles (**`numTrees`** entre 10 y 20) y niveles de profundidad (**`maxDepth`** entre 5 y 10).
3. Implementamos un **`CrossValidator`** con **`numFolds=3`**, lo que dividió y evaluó los datos en tres bloques distintos para asegurarnos matemáticamente de que el éxito del modelo es real.

In [5]:
# Calcular la proporción para balancear las clases
total_registros = df_muestra.count()
total_compras = df_muestra.filter(col("compro") == 1.0).count()
total_no_compras = total_registros - total_compras

# Calcular el peso inverso
peso_compra = total_registros / (2.0 * total_compras)
peso_no_compra = total_registros / (2.0 * total_no_compras)

# Añadir la columna de peso al DataFrame de entrenamiento
df_muestra = df_muestra.withColumn("weight", when(col("compro") == 1.0, peso_compra).otherwise(peso_no_compra))

# Transformar texto a números
brand_indexer = StringIndexer(inputCol="brand", outputCol="brand_index", handleInvalid="keep")
category_indexer = StringIndexer(inputCol="category_code", outputCol="category_index", handleInvalid="keep")

# Ensamblar el cector (
assembler = VectorAssembler(inputCols=["brand_index", "category_index", "price"],
                            outputCol="features",
                            handleInvalid="skip")

# Random Forest
rf = RandomForestClassifier(labelCol="compro", featuresCol="features", maxBins=1500, weightCol="weight")

pipeline = Pipeline(stages=[brand_indexer, category_indexer, assembler, rf])

print("Pipeline construido con éxito. Orden de las etapas:")
for stage in pipeline.getStages():
    print(f" - {type(stage).__name__}")

Pipeline construido con éxito. Orden de las etapas:
 - StringIndexer
 - StringIndexer
 - VectorAssembler
 - RandomForestClassifier


In [ ]:
# Definir los parámetros
paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [10, 20])     # Probará con 10 y con 20 árboles
             .addGrid(rf.maxDepth, [5, 10])      # Probará con profundidad 5 y 10
             .build())

# Área bajo la curva ROC
evaluator = BinaryClassificationEvaluator(labelCol="compro", metricName="areaUnderROC")

# CrossValidator
cv = CrossValidator(estimator=pipeline,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3, # Divide los datos en 3 pedazos
                    parallelism=2)

# Dividir en entrenamiento y prueba
train_df, test_df = df_muestra.randomSplit([0.8, 0.2], seed=42)
print("Iniciando el entrenamiento...")

cv_model = cv.fit(train_df)
print("¡Entrenamiento completado!")

Iniciando el entrenamiento...


In [ ]:
predicciones = cv_model.transform(test_df)

area_roc = evaluator.evaluate(predicciones)

print(f"Área ROC: {area_roc:.4f}")

# Extraer el mejor modelo
mejor_rf = cv_model.bestModel.stages[3]
print("Mejores hiperparámetros:")
print(f" - Número de árboles (numTrees): {mejor_rf.getNumTrees}")
print(f" - Profundidad máxima (maxDepth): {mejor_rf.getOrDefault('maxDepth')}")


print("\nComparando la realidad vs la predicción del modelo:")
predicciones.select("brand", "price", "compro", "prediction", "probability").show(10, truncate=False)

El modelo alcanzó un Área ROC de 0.6738. Esto demuestra que el algoritmo tiene una capacidad predictiva superior al 0.5. Tras la búsqueda con el optimizador, la combinación con mejor rendimiento requirió un numTrees de 20 y una maxDepth de 10. Indicando que el problema de negocio requiere árboles profundo.Al evaluar la tabla de salidas, se observa una alta coherencia en las predicciones. Por ejemplo, para marcas de alta gama como Apple o Longines asociadas a precios elevados.

# Aprendizaje no supervisado

Elegimos el algoritmo no supervisado **`KMeans`**  para realizar segmentaciones de mercado, ya que agrupa cientos de miles de registros basándose en distancias matemáticas.
Usamos **`StringIndexer`** (**`brand_indexer_km`** y **`category_indexer_km`**) y las unificamos junto al precio en un vector denso mediante **`VectorAssembler`** bajo la columna **`features`**, permitiendo que el modelo calcule las similitudes geométricas de los datos. Configuramos el algoritmo con **`k=3`** para identificar de forma natural tres perfiles económicos de clientes (Gama baja, media y alta), asignando los resultados en la columna **`cluster`** y fijando un **`seed=42`** para asegurar que los grupos se mantengan estables en cada ejecución. Finalmente, evaluamos la calidad de la separación con **`ClusteringEvaluator`** y (**`silhouette`**) y cruzamos los perfiles mediante **`groupBy("cluster")`** para extraer métricas de negocio  como el precio promedio y la tasa de conversión.

In [ ]:
brand_indexer_km = StringIndexer(inputCol="brand", outputCol="brand_index", handleInvalid="keep")
category_indexer_km = StringIndexer(inputCol="category_code", outputCol="category_index", handleInvalid="keep")

assembler_km = VectorAssembler(inputCols=["brand_index", "category_index", "price"],
                               outputCol="features",
                               handleInvalid="skip")

scaler = StandardScaler(inputCol="unscaled_features", outputCol="features", withStd=True, withMean=False)

kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)

pipeline_kmeans = Pipeline(stages=[brand_indexer_km, category_indexer_km, assembler_km, scaler, kmeans])

print("Entrenando el modelo No Supervisado K-Means...")

model_kmeans = pipeline_kmeans.fit(df_muestra)
print("¡Agrupación completada con éxito!")

In [ ]:
resultados_kmeans = model_kmeans.transform(df_muestra)

evaluator_km = ClusteringEvaluator(featuresCol="features", predictionCol="cluster", metricName="silhouette")
silhouette = evaluator_km.evaluate(resultados_kmeans)

print(f"Coeficiente de Silueta: {silhouette:.4f}")


resultados_kmeans.groupBy("cluster").agg(
    {"price": "avg", "compro": "avg", "cluster": "count"}
).withColumnRenamed("avg(price)", "Precio_Promedio") \
 .withColumnRenamed("avg(compro)", "Tasa_de_Compra") \
 .withColumnRenamed("count(cluster)", "Cantidad_de_Eventos") \
 .orderBy("cluster").show()

print("Muestra de los datos con su clúster asignado:")
resultados_kmeans.select("brand", "price", "event_type", "cluster").show(10, truncate=False)

Se obtuvo un coeficiente de silueta de 0.7112, indicando que los grupos están  separados y definidos:

Clúster 0 (consumo masivo / gama media): Reúne a la gran mayoría de la muestra con 241,791 eventos. Su precio promedio se sitúa en $218.90 y mantiene una  tasa de conversión del 17.91%. Es el núcleo de ingresos por volumen de la plataforma.

Clúster 1 (compradores premium / gama alta): Concentra 53,805 eventos orientados a marcas de gran valor con un precio promedio de $1059.21. Lo más valioso para el negocio es que posee la tasa de conversión más alta de la tienda (19.03%), lo que demuestra que los usuarios de este segmento entran a la plataforma con una alta intención de compra.

Clúster 2 (buscadores de ofertas / gama baja): Es el segmento menor con 19,796 eventos, caracterizado por un precio promedio de apenas $118.64 y la conversión más baja de 7.93%.